# IMDb Movie Recommender — Encodage & Choix de features (Notebook)

Objectif : **valider et optimiser l'encodage** (genres / acteurs / réalisateurs / scénaristes / genre_tokens) **avant** de figer `train.py`.

Contraintes :
- Pas d'affichage massif : `head(10)` max, échantillons limités.
- On prend des décisions **data-driven** (mesures + petits tests NN).


## 1) Setup

In [4]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from collections import Counter
import math

In [5]:
# --- Robust project root detection ---
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "config.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT =", PROJECT_ROOT)

sys.path.insert(0, str(PROJECT_ROOT))
import config as cfg

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

PROJECT_ROOT = /home/gau/projets/movie-recommender


## 2) Charger le dataset (parquet)

In [6]:
data_path = PROJECT_ROOT / cfg.DATA_PATH
print("Using:", data_path)

df = pd.read_parquet(data_path).reset_index(drop=True)
print("shape:", df.shape)

df.head(10)

Using: /home/gau/projets/movie-recommender/data/processed/movie_imdb_test.parquet
shape: (18089, 14)


,tconst,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,genre_tokens,actors,directors,writers,people,averageRating,numVotes,content
0,tt0004972,The Birth of a Nation,The Birth of a Nation,1915,195,Drama War,g:Drama g:War g:War g:War g:War gpair:Drama|War drama:bleak era:1910s format...,Lillian_Gish Mae_Marsh,D.W._Griffith,Thomas_Dixon_Jr. D.W._Griffith,D.W._Griffith Lillian_Gish Mae_Marsh Thomas_Dixon_Jr. D.W._Griffith,6.1,28078,The Birth of a Nation Drama War D.W._Griffith Lillian_Gish Mae_Marsh Thomas_...
1,tt0006864,Intolerance,Intolerance: Love's Struggle Throughout the Ages,1916,163,Drama History,g:Drama g:History g:History g:History gpair:Drama|History qual:high era:1910...,Lillian_Gish Lillian_Gish,D.W._Griffith,D.W._Griffith Anita_Loos,D.W._Griffith Lillian_Gish Lillian_Gish D.W._Griffith Anita_Loos,7.7,17814,Intolerance Drama History D.W._Griffith Lillian_Gish Lillian_Gish D.W._Griff...
2,tt0009968,Broken Blossoms,Broken Blossoms or The Yellow Man and the Girl,1919,90,Drama Romance,g:Drama g:Romance g:Romance gpair:Drama|Romance drama:romantic era:1910s for...,Lillian_Gish Richard_Barthelmess,D.W._Griffith,Thomas_Burke D.W._Griffith,D.W._Griffith Lillian_Gish Richard_Barthelmess Thomas_Burke D.W._Griffith,7.2,11712,Broken Blossoms Drama Romance D.W._Griffith Lillian_Gish Richard_Barthelmess...
3,tt0010323,The Cabinet of Dr. Caligari,Das Cabinet des Dr. Caligari,1920,67,Horror Mystery Thriller,g:Horror g:Horror g:Horror g:Horror g:Horror g:Mystery g:Mystery g:Mystery g...,Werner_Krauss Conrad_Veidt,Robert_Wiene,Carl_Mayer Hans_Janowitz,Robert_Wiene Werner_Krauss Conrad_Veidt Carl_Mayer Hans_Janowitz,8.0,75514,The Cabinet of Dr. Caligari Horror Mystery Thriller Robert_Wiene Werner_Krau...
4,tt0011130,Dr. Jekyll and Mr. Hyde,Dr. Jekyll and Mr. Hyde,1920,69,Drama Horror Sci-Fi,g:Drama g:Horror g:Horror g:Horror g:Horror g:Horror g:Sci-Fi g:Sci-Fi g:Sci...,John_Barrymore John_Barrymore,John_S._Robertson,Robert_Louis_Stevenson Clara_Beranger,John_S._Robertson John_Barrymore John_Barrymore Robert_Louis_Stevenson Clara...,6.9,6598,Dr. Jekyll and Mr. Hyde Drama Horror Sci-Fi John_S._Robertson John_Barrymore...
5,tt0011237,The Golem,"Der Golem, wie er in die Welt kam",1920,91,Fantasy Horror,g:Fantasy g:Fantasy g:Fantasy g:Fantasy g:Horror g:Horror g:Horror g:Horror ...,Paul_Wegener Paul_Wegener,Paul_Wegener,Paul_Wegener Henrik_Galeen,Paul_Wegener Paul_Wegener Paul_Wegener Paul_Wegener Henrik_Galeen,7.2,9411,The Golem Fantasy Horror Paul_Wegener Paul_Wegener Paul_Wegener Paul_Wegener...
6,tt0011841,Way Down East,Way Down East,1920,145,Drama Romance,g:Drama g:Romance g:Romance gpair:Drama|Romance drama:romantic era:1920s for...,Lillian_Gish Richard_Barthelmess,D.W._Griffith,Lottie_Blair_Parker William_A._Brady,D.W._Griffith Lillian_Gish Richard_Barthelmess Lottie_Blair_Parker William_A...,7.3,6259,Way Down East Drama Romance D.W._Griffith Lillian_Gish Richard_Barthelmess L...
7,tt0012349,The Kid,The Kid,1921,68,Comedy Drama Family,g:Comedy g:Comedy g:Drama g:Family g:Family g:Family gpair:Comedy|Drama gpai...,Charles_Chaplin Edna_Purviance,Charles_Chaplin,Charles_Chaplin,Charles_Chaplin Charles_Chaplin Edna_Purviance Charles_Chaplin,8.2,144556,The Kid Comedy Drama Family Charles_Chaplin Charles_Chaplin Edna_Purviance C...
8,tt0012364,The Phantom Carriage,Körkarlen,1921,107,Drama Fantasy Horror,g:Drama g:Fantasy g:Fantasy g:Fantasy g:Fantasy g:Horror g:Horror g:Horror g...,Victor_Sjöström Hilda_Borgström,Victor_Sjöström,Selma_Lagerlöf Victor_Sjöström,Victor_Sjöström Victor_Sjöström Hilda_Borgström Selma_Lagerlöf Victor_Sjöström,8.0,15563,The Phantom Carriage Drama Fantasy Horror Victor_Sjöström Victor_Sjöström Hi...
9,tt0012494,Destiny,Der müde Tod,1921,114,Drama Fantasy Horror,g:Drama g:Fantasy g:Fantasy g:Fantasy g:Fantasy g:Horror g:Horror g:Horror g...,Bernhard_Goetzke Bernhard_Goetzke,Fritz_Lang,Fritz_Lang Thea_von_Harbou,Fritz_Lang Bernhard_Goetzke Bernhard_Goetzke Fritz_Lang Thea_von_Harbou,7.6,7283,Destiny Drama

In [7]:
# Échantillon aléatoire (petit) pour inspection rapide
df.sample(6, random_state=42)[["primaryTitle","startYear","averageRating","numVotes","genres","directors","writers","actors","genre_tokens"]].head(10)

,primaryTitle,startYear,averageRating,numVotes,genres,directors,writers,actors,genre_tokens
2796,Losin' It,1982,5.0,5803,Comedy Drama,Curtis_Hanson,Bill_Norton Bryan_Gindoff,Tom_Cruise Jackie_Earle_Haley,g:Comedy g:Comedy g:Drama gpair:Comedy|Drama qual:low era:1980s format:long
12134,Violet & Daisy,2011,6.0,13734,Action Comedy Crime,Geoffrey_Fletcher,Geoffrey_Fletcher,Saoirse_Ronan Alexis_Bledel,g:Action g:Action g:Action g:Comedy g:Comedy g:Crime g:Crime g:Crime gpair:A...
15124,Samba,2014,6.7,17870,Comedy Drama Romance,Olivier_Nakache,Éric_Toledano Olivier_Nakache,Omar_Sy Charlotte_Gainsbourg,g:Comedy g:Comedy g:Drama g:Romance g:Romance gpair:Comedy|Drama gpair:Comed...
3287,Bloodsport,1988,6.8,102006,Action Biography Drama,Newt_Arnold,Sheldon_Lettich Christopher_Cosby,Jean-Claude_Van_Damme Donald_Gibb,g:Action g:Action g:Action g:Biography g:Biography g:Biography g:Drama gpair...
5858,The Wounds,1998,8.0,12425,Comedy Crime Drama,Srdjan_Dragojevic,Srdjan_Dragojevic,Dusan_Pekic Milan_Maric,g:Comedy g:Comedy g:Crime g:Crime g:Crime g:Drama gpair:Comedy|Crime gpair:C...
15284,Prem Ratan Dhan Payo,2015,4.4,25797,Action Drama Musical,Sooraj_R._Barjatya,Sooraj_R._Barjatya Aash_Karan_Atal,Salman_Khan Salman_Khan,g:Action g:Action g:Action g:Drama g:Musical g:Musical g:Musical gpair:Actio...


## 3) Sanity checks rapides

In [8]:
needed_cols = ["primaryTitle","startYear","averageRating","numVotes","genres","genre_tokens","directors","writers","actors"]
missing = [c for c in needed_cols if c not in df.columns]
print("Missing columns:", missing)

df[["averageRating","numVotes","runtimeMinutes"]].describe(percentiles=[.5,.75,.9,.95,.99])

Missing columns: []


,averageRating,numVotes,runtimeMinutes
count,18089.000000,18089.0,18089.0
mean,6.477323,63057.649068,109.004975
std,1.044464,150751.227919,22.097623
min,1.000000,5000.0,43.0
50%,6.600000,16245.0,104.0
75%,7.200000,49453.0,119.0
90%,7.700000,149073.6,138.0
95%,8.000000,270365.4,153.0
99%,8.400000,747807.16,179.0
max,9.300000,3150021.0,330.0


## 4) Choisir `min_votes` intelligemment

In [20]:
import numpy as np
import pandas as pd
from datetime import datetime

# ---------------------------
# 4.1) Features: âge + votes/an + version bayésienne
# ---------------------------
def add_votes_time_features(df: pd.DataFrame, current_year: int | None = None, tau_years: float = 3.0):
    """
    Ajoute :
      - age_years
      - votes_per_year
      - votes_per_year_bayes  (lissé bayésien, anti-biais films récents)
    tau_years = "années fictives" (prior). 3.0 est un bon défaut.
    """
    out = df.copy()

    if current_year is None:
        current_year = datetime.now().year

    year = pd.to_numeric(out["startYear"], errors="coerce")
    votes = pd.to_numeric(out["numVotes"], errors="coerce").fillna(0).astype(float)

    # âge >= 1 (on inclut l'année de sortie) pour éviter division par 0
    age = (current_year - year).clip(lower=0)
    age = (age + 1).fillna(1).astype(float)

    out["age_years"] = age
    out["votes_per_year"] = votes / out["age_years"]

    # prior global sur votes_per_year
    mu_vpy = float(out["votes_per_year"].median()) if len(out) else 0.0

    # Bayes smoothing:
    # votes_per_year_bayes = (mu*tau + votes) / (age + tau)
    out["votes_per_year_bayes"] = (mu_vpy * tau_years + votes) / (out["age_years"] + tau_years)

    # log versions (robustes)
    out["log_votes"] = np.log1p(votes)
    out["log_vpy"] = np.log1p(out["votes_per_year"])
    out["log_vpy_bayes"] = np.log1p(out["votes_per_year_bayes"])

    return out, mu_vpy


# ---------------------------
# 4.2) Suggestion "intelligente" de seuil
# ---------------------------
def suggest_threshold(
    series: pd.Series,
    thresholds=None,
    w_coverage: float = 0.50,
    w_size: float = 0.20,
    w_quality: float = 0.30,
):
    """
    Choisit un seuil t sur `series` en optimisant :
      - coverage (somme conservée / somme totale)
      - size_ratio (#items conservés / total)
      - quality (moyenne conservée, normalisée)
    """
    x = pd.to_numeric(series, errors="coerce").fillna(0).to_numpy(dtype=float)
    n_total = len(x)
    total_signal = float(x.sum())
    xmax = float(x.max()) if float(x.max()) > 0 else 1.0

    if thresholds is None:
        xpos = x[x > 0]
        lo = float(np.percentile(xpos, 5)) if len(xpos) else 0.1
        hi = float(np.percentile(xpos, 99.5)) if len(xpos) else 10.0
        hi = max(hi, lo * 1.01)

        # seuils log-spaced + quelques valeurs humaines
        thresholds = np.unique(np.round(np.logspace(np.log10(max(lo, 1e-6)), np.log10(hi), 40), 4)).tolist()
        thresholds = sorted(set(thresholds + [0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100]))

    rows = []
    for t in thresholds:
        mask = x >= t
        n = int(mask.sum())
        size_ratio = n / n_total if n_total else 0.0

        kept_signal = float(x[mask].sum()) if n else 0.0
        coverage = kept_signal / total_signal if total_signal > 0 else 0.0

        mean_kept = float(x[mask].mean()) if n else 0.0
        mean_norm = float(np.log1p(mean_kept) / np.log1p(xmax))

        score = w_coverage * coverage + w_size * size_ratio + w_quality * mean_norm

        rows.append({
            "threshold": float(t),
            "n_movies": n,
            "size_ratio": size_ratio,
            "coverage": coverage,
            "mean_kept": mean_kept,
            "score": score,
        })

    tbl = pd.DataFrame(rows).sort_values("threshold").reset_index(drop=True)

    # --- ignore plateau where nothing is filtered ---
    tbl_eff = tbl[tbl["size_ratio"] < 0.999].copy()
    if not tbl_eff.empty:
        tbl = tbl_eff
    best = float(tbl.loc[tbl["score"].idxmax(), "threshold"])
    return best, tbl



# ---------------------------
# 4.3) RUN
# ---------------------------
df_v, mu_vpy = add_votes_time_features(df, tau_years=3.0)
print("mu votes_per_year =", round(mu_vpy, 3))

# A) seuil brut sur numVotes (en log pour stabilité)
best_votes_log, tbl_votes_log = suggest_threshold(df_v["log_votes"])
print("best threshold on log_votes =", best_votes_log)

# B) seuil sur votes/an (bayésien) -> recommandé pour ne pas pénaliser les films récents
best_vpy_bayes, tbl_vpy_bayes = suggest_threshold(df_v["votes_per_year_bayes"])
print("best threshold on votes_per_year_bayes =", best_vpy_bayes)

mu votes_per_year = 1104.6
best threshold on log_votes = 8.6071
best threshold on votes_per_year_bayes = 100.0


In [10]:
tbl_vpy_bayes["no_filter"] = (tbl_vpy_bayes["size_ratio"] >= 0.9999)
tbl_vpy_bayes.sort_values("score", ascending=False).head(15)[
    ["threshold","n_movies","size_ratio","no_filter","coverage","mean_kept","score"]
]

,threshold,n_movies,size_ratio,no_filter,coverage,mean_kept,score
9,100.0000,18026,0.996517,False,0.999912,3670.419006,0.905358
10,175.7924,17184,0.949970,False,0.998105,3843.308568,0.896301
11,203.8215,16784,0.927857,False,0.996959,3930.382699,0.891867
12,236.3197,16383,0.905689,False,0.995625,4021.196839,0.887340
13,273.9995,15932,0.880756,False,0.993884,4127.800572,0.882140
14,317.6872,15442,0.853668,False,0.991694,4249.395891,0.876356
15,368.3406,14851,0.820996,False,0.988635,4404.875902,0.869195
16,427.0705,14216,0.785892,False,0.984814,4583.845180,0.861263
17,495.1645,13512,0.746973,False,0.979916,4798.686121,0.852180
18,574.1156,12755,0.705125,False,0.973807,5051.795448,0.842046


In [11]:
min_votes = 200
mask = pd.to_numeric(df["numVotes"], errors="coerce").fillna(0) >= min_votes
print("n_movies kept:", int(mask.sum()), "| ratio:", round(mask.mean(), 3))

n_movies kept: 18089 | ratio: 1.0


In [12]:
print("Chosen min_votes (final):", best_vpy_bayes)

Chosen min_votes (final): 100.0


## Section 4 — Dataset filtering (min_votes & temporal bias)

### Objective
Define a **robust filtering strategy** that:
- removes extremely obscure movies (noise),
- preserves diversity,
- does **not penalize recent movies** that did not yet have time to accumulate votes.

---

### Base filter: minimum number of votes

We fix a base threshold:

- **min_votes = 200**

Rationale:
- standard IMDb-like threshold,
- removes extreme noise,
- keeps ~99k movies (large diversity),
- stable baseline for the recommender.

---

### Temporal bias issue

Raw `numVotes` unfairly penalizes:
- recent releases,
- niche movies with strong early engagement.

To correct this, we introduce **votes per year**, smoothed with a Bayesian prior.

---

### Bayesian votes-per-year smoothing

We define:

- `age_years = current_year - startYear + 1`
- `votes_per_year = numVotes / age_years`

To avoid inflation for very recent movies, we apply Bayesian smoothing:

\[
votes\_per\_year\_bayes =
\frac{\mu_{vpy} \cdot \tau + numVotes}{age\_years + \tau}
\]

Where:
- \(\mu_{vpy}\) is the **median** votes-per-year over the dataset,
- \(\tau = 3\) years is a temporal prior.

Using the median ensures robustness to blockbusters and outliers.

---

### Final filtering rule

A movie is kept if **at least one** condition is satisfied:

- `numVotes >= 200`
- `votes_per_year_bayes >= 5`

Rationale:
- protects recent movies with real traction,
- preserves long-tail diversity,
- removes films that are both old **and** invisible.

---

### Final parameters (Section 4)

```text
MIN_VOTES = 200
VOTES_PER_YEAR_BAYES_TAU = 3.0
MIN_VOTES_PER_YEAR_BAYES = 5.0


## 5) Choisir `RATING_PRIOR` et `CONFIDENCE_K` (data-driven)

In [13]:
def suggest_prior_k(df: pd.DataFrame, q_prior: float = 0.50, q_k: float = 0.50, floor: int = 50):
    votes = pd.to_numeric(df["numVotes"], errors="coerce").fillna(0).to_numpy(dtype=float)
    votes = votes[votes > 0]
    if len(votes) == 0:
        return 150, 150
    prior = int(max(floor, np.quantile(votes, q_prior)))
    k = int(max(floor, np.quantile(votes, q_k)))
    return prior, k

prior_med, k_med = suggest_prior_k(df, 0.50, 0.50)
prior_strict, k_strict = suggest_prior_k(df, 0.60, 0.75)
prior_med, k_med, prior_strict, k_strict

(16245, 16245, 23838, 49453)

In [14]:
# Visual check rapide (pas indispensable)
global_mean = float(pd.to_numeric(df["averageRating"], errors="coerce").fillna(0).mean())
votes = pd.to_numeric(df["numVotes"], errors="coerce").fillna(0).astype(float)
ratings = pd.to_numeric(df["averageRating"], errors="coerce").fillna(0).astype(float)

def add_bayes_conf(df_in: pd.DataFrame, prior: float, k: float) -> pd.DataFrame:
    out = df_in.copy()
    out["rating_bayes"] = (global_mean * prior + votes * ratings) / (prior + votes)
    out["confidence"] = votes / (votes + k)
    return out

df_bc = add_bayes_conf(df, prior=prior_med, k=k_med)
df_bc[["averageRating","rating_bayes","confidence","numVotes"]].describe().round(3)

,averageRating,rating_bayes,confidence,numVotes
count,18089.000,18089.000,18089.000,18089.0
mean,6.477,6.527,0.548,63057.649
std,1.044,0.597,0.232,150751.228
min,1.000,1.956,0.235,5000.0
25%,5.900,6.209,0.336,8212.0
50%,6.600,6.524,0.500,16245.0
75%,7.200,6.837,0.753,49453.0
max,9.300,9.286,0.995,3150021.0


### Decision (Section 5)

We fix:
- RATING_PRIOR = 500
- CONFIDENCE_K = 500

Rationale:
- Values are close to dataset medians (~850 votes) but slightly lower to preserve diversity.
- rating_bayes smooths extremes without collapsing variance.
- confidence correctly reflects reliability from 200 votes upward.


## 6) Diagnostic tokens (diversité / entropie)

In [15]:
def entity_token_stats(series: pd.Series, name: str, top_k: int = 20):
    tokens = []
    for x in series.fillna("").astype(str):
        tokens.extend(x.split())

    c = Counter(tokens)
    total = sum(c.values()) if c else 1

    freqs = np.array(list(c.values()), dtype=float)
    probs = freqs / total

    entropy = float(-(probs * np.log2(probs + 1e-12)).sum())
    top_share = float(freqs.max() / total) if len(freqs) else 0.0
    uniq = len(c)
    eff_vocab = float(2 ** entropy)  # "effective" nb de tokens

    print(f"\n{name}")
    print("-" * 60)
    print(f"Total tokens: {total:,}")
    print(f"Unique tokens: {uniq:,}")
    print(f"Entropy (Shannon): {entropy:.2f}  |  Effective vocab: {eff_vocab:,.0f}")
    print(f"Top-1 share: {top_share*100:.3f}%")
    print(f"Top {top_k} tokens:")
    for t, v in c.most_common(top_k):
        print(f"  {t}: {v}")

for col in ["directors", "writers", "actors"]:
    if col in df.columns:
        entity_token_stats(df[col], col, top_k=15)


directors
------------------------------------------------------------
Total tokens: 18,089
Unique tokens: 7,301
Entropy (Shannon): 12.21  |  Effective vocab: 4,726
Top-1 share: 0.282%
Top 15 tokens:
  Woody_Allen: 51
  Alfred_Hitchcock: 42
  Clint_Eastwood: 40
  Steven_Spielberg: 33
  Steven_Soderbergh: 33
  Ridley_Scott: 29
  John_Huston: 28
  Martin_Scorsese: 27
  Ron_Howard: 27
  Sidney_Lumet: 26
  John_Ford: 25
  Tyler_Perry: 25
  Billy_Wilder: 24
  Akira_Kurosawa: 24
  Brian_De_Palma: 24

writers
------------------------------------------------------------
Total tokens: 30,104
Unique tokens: 15,700
Entropy (Shannon): 13.45  |  Effective vocab: 11,184
Top-1 share: 0.176%
Top 15 tokens:
  Woody_Allen: 53
  Stephen_King: 52
  Luc_Besson: 46
  William_Shakespeare: 36
  David_Koepp: 28
  Billy_Wilder: 27
  John_Hughes: 27
  Tyler_Perry: 26
  Ethan_Coen: 24
  Pedro_Almodóvar: 23
  John_Carpenter: 22
  Robert_Mark_Kamen: 22
  Akira_Kurosawa: 21
  Joel_Coen: 21
  Sylvester_Stallone: 20


In [16]:
def binary_entropy(p: float) -> float:
    if p <= 0 or p >= 1:
        return 0.0
    return float(-(p*math.log(p, 2) + (1-p)*math.log(1-p, 2)))

def genre_token_df_stats(series: pd.Series, top_k: int = 20, prefix_filter: str | None = None):
    docs = series.fillna("").astype(str).apply(lambda s: set(s.split()))
    N = len(docs)

    df_counts = Counter()
    for toks in docs:
        if prefix_filter:
            toks = {t for t in toks if t.startswith(prefix_filter)}
        for t in toks:
            df_counts[t] += 1

    stats = (
        pd.DataFrame({"token": list(df_counts.keys()), "df": list(df_counts.values())})
          .assign(p=lambda x: x["df"] / N)
    )
    stats["H_bin"] = stats["p"].apply(binary_entropy)
    stats["idf"] = np.log((N + 1) / (stats["df"] + 1)) + 1

    # petit bonus : type de token
    def tok_type(t: str) -> str:
        return t.split(":", 1)[0] if ":" in t else "other"
    stats["type"] = stats["token"].apply(tok_type)

    print("\ngenre_tokens — DF/IDF diagnostics" + (f" ({prefix_filter})" if prefix_filter else ""))
    print("-" * 60)
    print(f"Nb films: {N:,} | Nb tokens distincts: {len(stats):,}")

    # top polluants (DF haut)
    print("\nTop tokens (DF highest) — 'polluants':")
    display(stats.sort_values("df", ascending=False).head(top_k))

    # top signatures (IDF haut)
    print("\nTop tokens (IDF highest) — 'signatures':")
    display(stats.sort_values("idf", ascending=False).head(top_k))

    # par type
    print("\nRépartition par type:")
    display(stats.groupby("type").agg(
        tokens=("token","count"),
        avg_df=("df","mean"),
        avg_idf=("idf","mean")
    ).sort_values("tokens", ascending=False))

    return stats

stats_all = genre_token_df_stats(df["genre_tokens"], top_k=15)

# Optionnel : focus uniquement sur les genres simples/pairs
stats_g     = genre_token_df_stats(df["genre_tokens"], top_k=15, prefix_filter="g:")
stats_gpair = genre_token_df_stats(df["genre_tokens"], top_k=15, prefix_filter="gpair:")



genre_tokens — DF/IDF diagnostics
------------------------------------------------------------
Nb films: 18,089 | Nb tokens distincts: 291

Top tokens (DF highest) — 'polluants':


,token,df,p,H_bin,idf,type
2,format:long,18054,0.998065,0.020229,1.001937,format
3,g:Drama,10684,0.590635,0.976166,1.526518,g
34,g:Comedy,6472,0.357787,0.940830,2.027720,g
225,era:2010s,5440,0.300735,0.882188,2.201396,era
49,g:Action,4179,0.231024,0.779793,2.465048,g
150,era:2000s,3737,0.206590,0.734913,2.576809,era
39,g:Crime,3619,0.200066,0.722061,2.608885,g
9,g:Romance,3114,0.172149,0.662597,2.759130,g
15,g:Thriller,3008,0.166289,0.649145,2.793752,g
7,qual:high,2982,0.164852,0.645791,2.802430,qual



Top tokens (IDF highest) — 'signatures':


,token,df,p,H_bin,idf,type
290,gpair:History|Sci-Fi,1,0.000055,0.000862,10.109967,gpair
266,gpair:Musical|Sport,1,0.000055,0.000862,10.109967,gpair
265,gpair:Fantasy|Sport,1,0.000055,0.000862,10.109967,gpair
42,gpair:Documentary|Fantasy,1,0.000055,0.000862,10.109967,gpair
103,gpair:Adventure|Film-Noir,1,0.000055,0.000862,10.109967,gpair
288,gpair:Music|Sci-Fi,1,0.000055,0.000862,10.109967,gpair
280,gpair:Crime|War,1,0.000055,0.000862,10.109967,gpair
164,gpair:Film-Noir|Music,1,0.000055,0.000862,10.109967,gpair
178,gpair:Action|Film-Noir,1,0.000055,0.000862,10.109967,gpair
157,gpair:Comedy|Film-Noir,1,0.000055,0.000862,10.109967,gpair



Répartition par type:


...

,token,df,p,H_bin,idf,type
10,gpair:Comedy|Drama,2675,0.147880,0.604509,2.911036,gpair
2,gpair:Drama|Romance,2268,0.125380,0.544630,3.076020,gpair
42,gpair:Crime|Drama,2210,0.122174,0.535577,3.101914,gpair
40,gpair:Action|Drama,1596,0.088230,0.430535,3.427232,gpair
21,gpair:Comedy|Romance,1591,0.087954,0.429603,3.430368,gpair
25,gpair:Action|Adventure,1344,0.074299,0.381767,3.598965,gpair
58,gpair:Action|Crime,1320,0.072973,0.376920,3.616970,gpair
33,gpair:Drama|Thriller,1308,0.072309,0.374482,3.626096,gpair
41,gpair:Drama|Mystery,1053,0.058212,0.320307,3.842767,gpair
49,gpair:Biography|Drama,974,0.053845,0.302510,3.920677,gpair



Top tokens (IDF highest) — 'signatures':


,token,df,p,H_bin,idf,type
17,gpair:Documentary|Fantasy,1,0.000055,0.000862,10.109967,gpair
62,gpair:Adventure|Film-Noir,1,0.000055,0.000862,10.109967,gpair
206,gpair:History|Sci-Fi,1,0.000055,0.000862,10.109967,gpair
203,gpair:Animation|Thriller,1,0.000055,0.000862,10.109967,gpair
204,gpair:Music|Sci-Fi,1,0.000055,0.000862,10.109967,gpair
205,gpair:Horror|Sport,1,0.000055,0.000862,10.109967,gpair
202,gpair:Documentary|Thriller,1,0.000055,0.000862,10.109967,gpair
183,gpair:Fantasy|Sport,1,0.000055,0.000862,10.109967,gpair
184,gpair:Musical|Sport,1,0.000055,0.000862,10.109967,gpair
120,gpair:Action|Film-Noir,1,0.000055,0.000862,10.109967,gpair



Répartition par type:


,tokens,avg_df,avg_idf
type,,,
gpair,207,190.164251,7.278908


In [17]:
def tokens_per_movie(series: pd.Series, name: str):
    counts = series.fillna("").astype(str).apply(lambda s: len(s.split()))
    print(f"\n{name} — tokens par film")
    print("-" * 60)
    display(counts.describe(percentiles=[.5, .75, .9, .95, .99]).to_frame("value"))

tokens_per_movie(df["genre_tokens"], "genre_tokens")
tokens_per_movie(df["actors"], "actors")
tokens_per_movie(df["directors"], "directors")
tokens_per_movie(df["writers"], "writers")



genre_tokens — tokens par film
------------------------------------------------------------


,value
count,18089.000000
mean,12.948975
std,5.165718
min,3.000000
50%,13.000000
75%,17.000000
90%,20.000000
95%,21.000000
99%,24.000000
max,25.000000



actors — tokens par film
------------------------------------------------------------


,value
count,18089.000000
mean,1.995799
std,0.064684
min,1.000000
50%,2.000000
75%,2.000000
90%,2.000000
95%,2.000000
99%,2.000000
max,2.000000



directors — tokens par film
------------------------------------------------------------


,value
count,18089.0
mean,1.0
std,0.0
min,1.0
50%,1.0
75%,1.0
90%,1.0
95%,1.0
99%,1.0
max,1.0



writers — tokens par film
------------------------------------------------------------


,value
count,18089.000000
mean,1.664216
std,0.482239
min,0.000000
50%,2.000000
75%,2.000000
90%,2.000000
95%,2.000000
99%,2.000000
max,2.000000


## 7) Tests NN par bloc (feature seule)

In [18]:
from collections import Counter

def entity_token_stats(series: pd.Series, name: str, top_k: int = 20):
    tokens = []
    for x in series.fillna("").astype(str):
        tokens.extend(x.split())

    c = Counter(tokens)
    total = sum(c.values()) if c else 1

    freqs = np.array(list(c.values()), dtype=float)
    probs = freqs / total

    entropy = float(-(probs * np.log2(probs + 1e-12)).sum())
    top_share = float(freqs.max() / total) if len(freqs) else 0.0
    uniq = len(c)
    eff_vocab = float(2 ** entropy)  # "effective" nb de tokens

    print(f"\n{name}")
    print("-" * 60)
    print(f"Total tokens: {total:,}")
    print(f"Unique tokens: {uniq:,}")
    print(f"Entropy (Shannon): {entropy:.2f}  |  Effective vocab: {eff_vocab:,.0f}")
    print(f"Top-1 share: {top_share*100:.3f}%")
    print(f"Top {top_k} tokens:")
    for t, v in c.most_common(top_k):
        print(f"  {t}: {v}")

for col in ["directors", "writers", "actors"]:
    if col in df.columns:
        entity_token_stats(df[col], col, top_k=15)


directors
------------------------------------------------------------
Total tokens: 18,089
Unique tokens: 7,301
Entropy (Shannon): 12.21  |  Effective vocab: 4,726
Top-1 share: 0.282%
Top 15 tokens:
  Woody_Allen: 51
  Alfred_Hitchcock: 42
  Clint_Eastwood: 40
  Steven_Spielberg: 33
  Steven_Soderbergh: 33
  Ridley_Scott: 29
  John_Huston: 28
  Martin_Scorsese: 27
  Ron_Howard: 27
  Sidney_Lumet: 26
  John_Ford: 25
  Tyler_Perry: 25
  Billy_Wilder: 24
  Akira_Kurosawa: 24
  Brian_De_Palma: 24

writers
------------------------------------------------------------
Total tokens: 30,104
Unique tokens: 15,700
Entropy (Shannon): 13.45  |  Effective vocab: 11,184
Top-1 share: 0.176%
Top 15 tokens:
  Woody_Allen: 53
  Stephen_King: 52
  Luc_Besson: 46
  William_Shakespeare: 36
  David_Koepp: 28
  Billy_Wilder: 27
  John_Hughes: 27
  Tyler_Perry: 26
  Ethan_Coen: 24
  Pedro_Almodóvar: 23
  John_Carpenter: 22
  Robert_Mark_Kamen: 22
  Akira_Kurosawa: 21
  Joel_Coen: 21
  Sylvester_Stallone: 20


## 8) Prototype encodage complet (comme `train.py`)

In [19]:
from scipy import sparse
from sklearn.preprocessing import MultiLabelBinarizer

def split_genres(series: pd.Series) -> list[list[str]]:
    return (
        series.fillna("")
        .astype(str)
        .str.replace(",", " ", regex=False)
        .str.strip()
        .str.split()
        .tolist()
    )

# Suggested TF-IDF params (adjust based on block tests)
params_directors = dict(min_df=2, max_features=5000)
params_writers   = dict(min_df=2, max_features=8000)
params_actors    = dict(min_df=10, max_features=15000)
params_gtokens   = dict(min_df=1, max_features=5000)

# Suggested weights (tune as needed)
W = dict(genres=1.0, directors=2.0, writers=1.3, actors=0.6, genre_tokens=0.8)

# Encoders
mlb = MultiLabelBinarizer(sparse_output=True)
X_genres = mlb.fit_transform(split_genres(df["genres"]))

vec_dir = TfidfVectorizer(lowercase=False, token_pattern=r"(?u)\b\w+\b", **params_directors)
vec_wri = TfidfVectorizer(lowercase=False, token_pattern=r"(?u)\b\w+\b", **params_writers)
vec_act = TfidfVectorizer(lowercase=False, token_pattern=r"(?u)\b\w+\b", **params_actors)
vec_gt  = TfidfVectorizer(lowercase=False, token_pattern=r"(?u)\b\w+\b", **params_gtokens)

X_dir = vec_dir.fit_transform(df["directors"].fillna("").astype(str))
X_wri = vec_wri.fit_transform(df["writers"].fillna("").astype(str))
X_act = vec_act.fit_transform(df["actors"].fillna("").astype(str))
X_gt  = vec_gt.fit_transform(df["genre_tokens"].fillna("").astype(str))

X = sparse.hstack([
    W["genres"]       * X_genres,
    W["directors"]    * X_dir,
    W["writers"]      * X_wri,
    W["actors"]       * X_act,
    W["genre_tokens"] * X_gt,
], format="csr")

print("Combined X shape:", X.shape)

NameError: name 'TfidfVectorizer' is not defined

In [ ]:
# Quick NN test on combined encoding
from sklearn.neighbors import NearestNeighbors

def recommend_from_X(df: pd.DataFrame, X, title: str, k: int = 10):
    q = title.strip().lower()
    sub = df[df["primaryTitle"].fillna("").astype(str).str.lower() == q]
    if sub.empty:
        sub = df[df["primaryTitle"].fillna("").astype(str).str.lower().str.contains(q, regex=False)]
    if sub.empty:
        print(f"❌ Query not found: {title}")
        return
    sub = sub.copy()
    sub["numVotes"] = pd.to_numeric(sub["numVotes"], errors="coerce").fillna(0)
    row = sub.sort_values("numVotes", ascending=False).iloc[0]
    idx = int(row.name)

    nn = NearestNeighbors(metric="cosine", n_neighbors=k+1)
    nn.fit(X)
    dist, ind = nn.kneighbors(X[idx])

    print(f"\nCombined encoding | query: {df.loc[idx,'primaryTitle']} ({df.loc[idx,'startYear']})")
    r = 0
    for d, i in zip(dist[0], ind[0]):
        if int(i) == idx:
            continue
        r += 1
        print(f"{r:2d}. {df.iloc[int(i)]['primaryTitle']} ({df.iloc[int(i)]['startYear']}) | sim={1-float(d):.3f}")
        if r >= k:
            break

recommend_from_X(df, X, "Inception", k=10)


Combined encoding | query: Inception (2010)
 1. Tenet (2020) | sim=0.848
 2. Interstellar (2014) | sim=0.798
 3. Dunkirk (2017) | sim=0.707
 4. The Prestige (2006) | sim=0.680
 5. The Dark Knight (2008) | sim=0.655
 6. The Dark Knight Rises (2012) | sim=0.655
 7. Roar: Tigers of the Sundarbans (2014) | sim=0.613
 8. My iz budushchego 2 (2010) | sim=0.613
 9. 2101 (2014) | sim=0.613
10. Mercury Man (2006) | sim=0.613
